<a href="https://colab.research.google.com/github/MRsources/MRzero-Core/blob/main/documentation/playground_mr0/Pulseq_zero_TSE_FAopt_SARtask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypulseq==1.5.0post1
!pip install pulseqzero==0.2.1
!pip install ismrmrd
!pip install MRzeroCore --no-deps &> /dev/null
!pip install bifti --no-deps &> /dev/null
!pip install pydisseqt

!wget https://github.com/MRsources/MRzero-Core/raw/main/documentation/playground_mr0/numerical_brain_cropped.mat &> /dev/null


(TSE_FAopt_SAR)=
# TSE flip angle opt. for SAR (with pulseq-zero)
Pulseq-zero combines Pusleq and MR-zero and allows you to optimize a Pulseq sequence dirrectly.

For example, herein we want to optimize the variable refocusing flip angle train of a TSE sequence to achieve low SAR.



###First,###
we now need to define a TSE sequence as function with parameters we wish to optimize as arguments:

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import cm

import MRzeroCore as mr0
import pulseqzero
pp = pulseqzero.pp_impl

In [ ]:
#@title Define TSE in PyPulseq
def tse_sequence(fov=200e-3, slice_thickness=8e-3, base_resolution=42,
                 TE_ms=5, TI_s=0, Ex_FA=90, Ref_FA=180, r_spoil=2,
                 PE_grad_on=True, RO_grad_on=True):
    """
    Generates a TSE sequence using PyPulseq.

    Args:
        system: PyPulseq system object.
        fov (float): Field of view in meters (default: 200e-3).
        slice_thickness (float): Slice thickness in meters (default: 8e-3).
        base_resolution (int): Base resolution for frequency and phase encoding (default: 42).
        TE_ms (float): Echo time in milliseconds (default: 5).
        TI_s (float): Inversion time in seconds (default: 0).
        Ex_FA (float): Excitation flip angle in degrees (default: 90).
        Ref_FA (float or array): Refocusing flip angle in degrees (default: 180).
        r_spoil (float): Spoil gradient factor (default: 2).
        PE_grad_on (bool): Enable/disable phase encoding gradients (default: True).
        RO_grad_on (bool): Enable/disable readout gradients (default: True).

    Returns:
        PyPulseq sequence object.
    """
    # Define resolution
    Nread = base_resolution  # frequency encoding steps/samples
    Nphase = base_resolution  # phase encoding steps/samples
    TE = TE_ms * 1e-3

    # %% S1. SETUP sys
    system = pp.Opts(
        max_grad=28, grad_unit='mT/m', max_slew=150, slew_unit='T/m/s',
        rf_ringdown_time=20e-6, rf_dead_time=100e-6,
        adc_dead_time=20e-6, grad_raster_time=10e-6)

    seq = pp.Sequence(system)

    # Define rf events
    rf1, gz1, gzr1 = pp.make_sinc_pulse(
        flip_angle=Ex_FA * np.pi / 180, phase_offset=90 * np.pi / 180, duration=1e-3,
        slice_thickness=slice_thickness, apodization=0.5, time_bw_product=4,
        system=system, return_gz=True)

    rf2, gz2, _ = pp.make_sinc_pulse(
        flip_angle=180*np.pi / 180, duration=1e-3,
        slice_thickness=slice_thickness, apodization=0.5, time_bw_product=4,
        system=system, return_gz=True)

    dwell=50e-6*2

    G_flag=(int(RO_grad_on),int(PE_grad_on))  # gradient flag (read,PE), if (0,0) all gradients are 0, for (1,0) PE is off

    # Define other gradients and ADC events
    gx = pp.make_trapezoid(channel='x', rise_time = 0.5*dwell, flat_area=Nread / fov*G_flag[0], flat_time=Nread*dwell, system=system)
    adc = pp.make_adc(num_samples=Nread, duration=Nread*dwell, phase_offset=90 * np.pi / 180, delay=0*gx.rise_time, system=system)
    gx_pre0 = pp.make_trapezoid(channel='x', area=+((1.0 + r_spoil) * gx.area / 2) , duration=1.5e-3, system=system)
    gx_prewinder = pp.make_trapezoid(channel='x', area=+(r_spoil * gx.area / 2), duration=1e-3, system=system)
    gp = pp.make_trapezoid(channel='y', area=0 / fov, duration=1e-3, system=system)
    rf_prep = pp.make_block_pulse(flip_angle=180 * np.pi / 180, duration=1e-3, system=system)


    # FLAIR
    if TI_s>0:
      seq.add_block(rf_prep)
      seq.add_block(pp.make_delay(TI_s))
      seq.add_block(gx_pre0)

    seq.add_block(rf1,gz1)
    seq.add_block(gx_pre0,gzr1)

    # the minimal TE is given by one full period form ref pulse to ref pulse, thus gz2+gx+2*gp
    minTE2=(pp.calc_duration(gz2) +pp.calc_duration(gx) + 2*pp.calc_duration(gp))/2

    minTE2=np.round(minTE2/10e-5)*10e-5


    # to realize longer TE,  we introduce a TEdelay that is added before and afetr the encoding period
    TEd=np.round(max(0, (TE/2-minTE2))/10e-5)*10e-5  # round to raster time

    if TEd==0:
      print('echo time set to minTE [ms]', 2*(minTE2 +TEd)*1000)
    else:
      print(' TE [ms]', 2*(minTE2 +TEd)*1000)


    # last timing step is to add TE/2 also between excitation and first ref pulse
    # from pulse top to pulse top we have already played out one full rf and gx_pre0, thus we substract these from TE/2
    seq.add_block(pp.make_delay((minTE2 +TEd ) - pp.calc_duration(gz1)-pp.calc_duration(gx_pre0)))

    encoding = []

    for echo_idx, ii in enumerate(range(-Nphase // 2, Nphase // 2)):  # e.g. -64:63
        gp  = pp.make_trapezoid(channel='y', area=+ii / fov*G_flag[1], duration=1e-3, system=system)
        gp_ = pp.make_trapezoid(channel='y', area=-ii / fov*G_flag[1], duration=1e-3, system=system)
        encoding.append(ii)

        # Try to index into a variable FA array, if it fails treat it as number
        try:
            flip_angle = Ref_FA[echo_idx] * torch.pi / 180
        except:
            flip_angle = Ref_FA * torch.pi / 180
        rf2, gz2, _ = pp.make_sinc_pulse(flip_angle=flip_angle, duration=1e-3,slice_thickness=slice_thickness, apodization=0.5, time_bw_product=4,system=system, return_gz=True)

        seq.add_block(rf2,gz2)
        seq.add_block(pp.make_delay(TEd)) # TE delay
        seq.add_block(gx_prewinder, gp)
        seq.add_block(adc, gx)
        seq.add_block(gx_prewinder, gp_)
        seq.add_block(pp.make_delay(TEd)) # TE delay

    # %% S2. CHECK, PLOT and WRITE the sequence  as .seq
    # Check whether the timing of the sequence is correct
    ok, error_report = seq.check_timing()
    if ok:
        print('Timing check passed successfully')
    else:
        print('Timing check failed. Error listing follows:')
        [print(e) for e in error_report]

    return seq, encoding

In [ ]:
# @title image reconstruction via FFT

def reconstruction(signal, encoding, Nread, Nphase):
  # reconstruct image
  kspace = torch.reshape((signal), (Nread, Nphase)).clone().t()
  encoding = np.stack(encoding)
  ipermvec = np.argsort(encoding)
  kspace=kspace[:,ipermvec]

  # fftshift iFFT fftshift
  spectrum = torch.fft.fftshift(kspace)
  space = torch.fft.ifft2(spectrum)
  space = torch.fft.ifftshift(space)

  return space

In [ ]:
# @title ploting functions for optimization steps and results

def plot_results_images(target, result, finished=False, colorbars=False):

  # show target, initial and optimized image on common colorscale
  vmin = min(target.min(), result.min())
  vmax = max(target.max(), result.max())

  plt.subplot(121)
  plt.title("optimizer target")
  plt.axis('off')
  mr0.util.imshow(target, vmin=vmin, vmax=vmax, cmap=cm.gray)
  if colorbars: plt.colorbar(cmap='gray')

  plt.subplot(122)
  if finished: plt.title("optimizer result")
  else: plt.title("optimizer step")
  plt.axis('off')
  mr0.util.imshow(result, vmin=vmin, vmax=vmax, cmap=cm.gray)
  if colorbars: plt.colorbar(cmap='gray')

  plt.show()

def plot_optimizer_history(loss_hist, param_hist, rSAR_hist, finished=False):
    #plt.figure(figsize=(12, 4))

    plt.subplot(121)
    plt.title("Loss and rSAR")
    plt.xlabel("itertation")
    plt.plot([l / loss_hist[0] for l in loss_hist], label="loss")
    plt.plot(rSAR_hist, label="rSAR")
    if finished:
      plt.plot([np.argmin(loss_hist), np.argmin(loss_hist)], [[l / loss_hist[0] for l in loss_hist][np.argmin(loss_hist)], rSAR_hist[np.argmin(loss_hist)]], "rx", label="optimum")

    plt.legend()
    plt.grid()

    plt.subplot(122)
    plt.xlabel("repetition")
    plt.ylabel("FA")
    plt.title("Optim. param")
    if finished: plt.plot(np.array(param_hist).T)
    else: plt.plot(np.array(param_hist[-2:]).T) # only plot current and last flip angle configuration
    plt.gca().yaxis.tick_right()
    plt.grid()

    plt.show()

def plot_optimized_flipangles(fa_optimized):
  plt.plot(fa_optimized, "o--")
  plt.xlabel("repetition")
  plt.ylabel("FA [deg]")
  plt.title("Optimized refocusing flip angles")
  plt.gca().yaxis.tick_right()
  plt.grid()
  plt.show()

In [ ]:
# @title Setup spin system

obj_p = mr0.util.load_phantom([96,96])


In [ ]:
# @title Generate optimization target

base_resolution=42
Ex_FA=90    # excitation flip angle
Ref_FA_target= torch.full((base_resolution,), 180)  # refocusing flip angle

with pulseqzero.mr0_mode():
  seq, encoding = tse_sequence(base_resolution=base_resolution, Ex_FA=Ex_FA, Ref_FA=Ref_FA_target)

  seq0 = seq.to_mr0()
  signal,_ = mr0.util.simulate(seq0,obj_p,accuracy=1e-4)

# reconstruct image
space = reconstruction(signal, encoding, base_resolution, base_resolution)

# plot result
plt.subplot(121)
plt.title('FFT-magnitude')
mr0.util.imshow(np.abs(space.numpy()), cmap=cm.gray)
plt.colorbar()

# store target for optimization
target = torch.abs(space)

In [ ]:
# @title Perform optimization

# sequence parametes
base_resolution=42
Ex_FA = 90    # excitation flip angle
Ref_FA = torch.full((base_resolution,), 180.0, requires_grad=True)  # refocusing flip angles

# initalize optimizer
iterations = 100
params = [{"params": Ref_FA, "lr": 5.0}]  # adjust learning rate as needed
optimizer = torch.optim.Adam(params)

lambda_SAR = 0.15
lambda_image = 1-lambda_SAR

loss_hist = []
rSAR_hist = []
Ref_FA_hist = []

# optimization loop
for i in range(iterations):

    optimizer.zero_grad()

    # ====
    # simulate
    # ====

    with pulseqzero.mr0_mode():
      seq, encoding = tse_sequence(base_resolution=base_resolution, Ex_FA=Ex_FA, Ref_FA=Ref_FA)

      seq0 = seq.to_mr0()

      if i%5 == 0:
        graph = mr0.compute_graph(seq0, obj_p.build(), 100000, 1e-4)

      signal = mr0.execute_graph(graph, seq0, obj_p.build(), 1e-4, 1e-4)
    # reconstruct image
    space = reconstruction(signal, encoding, base_resolution, base_resolution)
    image = torch.abs(space) # current optimizer step image


    # ====
    # loss computation
    # ====

    MSE_image = ((image - target)**2).mean()/((target**2).mean())  # MSE of images
    rSAR = torch.sum(Ref_FA**2)/torch.sum(Ref_FA_target**2)     # relative SAR

    loss = lambda_SAR * rSAR + lambda_image * MSE_image

    print(f"{i+1} / {iterations}: loss={loss.item()}, rSAR={rSAR}, Ref_FA={Ref_FA.detach().numpy()}")

    loss_hist.append(loss.item())
    rSAR_hist.append(rSAR.item())
    Ref_FA_hist.append(Ref_FA.detach().numpy().copy())

    # ====
    # perform optimizer step
    # ====

    loss.backward()
    optimizer.step()

    # plot images
    plot_results_images(target, image)

    # optimization timeline
    plot_optimizer_history(loss_hist, Ref_FA_hist, rSAR_hist)

In [ ]:
# @title Evaluate optimization result


# simute optimizer result: optimal flip angle configuration
with pulseqzero.mr0_mode():
  seq, encoding = tse_sequence(base_resolution=base_resolution, Ex_FA=Ex_FA, Ref_FA=Ref_FA_hist[np.argmin(loss_hist)])
  seq0 = seq.to_mr0()
  graph = mr0.compute_graph(seq0, obj_p.build(), 100000, 1e-8)
  signal = mr0.execute_graph(graph, seq0, obj_p.build(), 1e-8, 1e-8)  # high accuracy to check if more states are neccesary

  # reconstruct image
  space = reconstruction(signal, encoding, base_resolution, base_resolution)
  result = torch.abs(space) # current optiumizer step image

# ====
# plot results
# ====

print(f"Optimized rSNR: {rSAR_hist[np.argmin(loss_hist)]}")
print(f"Optimized Ref_FA: {Ref_FA_hist[np.argmin(loss_hist)]}")

# images
plot_results_images(target, result, finished=True)

# optimization timeline
plot_optimizer_history(loss_hist, Ref_FA_hist, rSAR_hist, finished=True)

# optimized flip angle configuration
plot_optimized_flipangles(Ref_FA_hist[np.argmin(loss_hist)])